# 🧪 LocateAnything-3B · Test đếm SẢN PHẨM (supervision, đẹp + chuẩn)

Test **open-vocab** của LocateAnything-3B trên **3 video thật**, gán nhãn bằng
**supervision**: box bo góc + nhãn + trace + **LineZone** (đếm cắt vạch) +
**PolygonZone** (khoanh vùng băng chuyền, đếm số vật trong vùng).

| Video | Bài toán | Vạch | Vùng polygon |
|---|---|---|---|
| `packages_rollers.mp4` | đếm **package** (con lăn) | ngang y≈0.65 | ôm băng con lăn |
| `packages_belt.mp4` | đếm **package** (có nhãn) | ngang y≈0.60 | ôm mặt belt |
| `tomatoes_sorting.mp4` | đếm **cà chua** | ngang y≈0.72 | ôm các làn |

Vật đi **xuống/về phía camera** → vạch NGANG. Mỗi video chạy vài **query KHÓ**
(mô tả bằng lời + tiếng Việt). Tối ưu tốc độ: ảnh nhỏ · ít token · lấy mẫu thưa ·
**nạp model 1 LẦN**. Video nằm sẵn trong repo → không cần upload.

In [ ]:
# ⚙️ Cài đặt: clone repo (kèm video) + phụ thuộc + kiểm tra GPU
import os, sys, subprocess

def sh(*a):
    print("$", " ".join(a)); subprocess.run(list(a), check=True)

if os.path.isdir("/kaggle/working"): WORK = "/kaggle/working"
elif os.path.isdir("/content"):      WORK = "/content"
else:                                 WORK = os.getcwd()
os.chdir(WORK); print("WORK =", WORK)

BRANCH = "claude/locate-anything-test-suite-xwju2f"
URL    = "https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git"
REPO   = os.path.join(WORK, "VisionOS")
if not os.path.isdir(os.path.join(REPO, ".git")):
    sh("git", "clone", "--depth", "1", "--branch", BRANCH, URL, REPO)
else:
    sh("git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH)
    sh("git", "-C", REPO, "reset", "--hard", "origin/" + BRANCH)

CODE = os.path.join(REPO, "VisionOS")          # code + sample_videos/ ở đây
os.chdir(CODE); sys.path.insert(0, CODE)
print("CODE =", CODE)

# LocateAnything-3B CẦN transformers==4.57.1
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.57.1", "accelerate", "supervision",
                "eva-decord", "lmdb"], check=False)

# xoá cache module (để chạy lại lấy code mới nhất)
for m in [m for m in list(sys.modules)
          if m.split(".")[0] in ("la_counting", "recognition", "run_la_conveyor")]:
    del sys.modules[m]

try:
    import torch
    print("CUDA:", torch.cuda.is_available(),
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except Exception as e:
    print("torch:", e)

In [ ]:
# 🎯 Cấu hình 3 video: vạch + polygon + query KHÓ; xem trước (không cần GPU)
import cv2, numpy as np, matplotlib.pyplot as plt

VID = os.path.join(CODE, "sample_videos")
VIDEOS = [
  {"name": "Package · con lăn", "task": "package",
   "path": os.path.join(VID, "packages_rollers.mp4"),
   "orient": "horizontal", "line_pos": 0.65,
   "polygon": [(0.02, 0.60), (0.98, 0.52), (0.99, 0.92), (0.05, 0.99)],
   "queries": ["a cardboard box on the roller conveyor",
               "a sealed shipping package",
               "kiện hàng carton"]},
  {"name": "Package · có nhãn", "task": "package",
   "path": os.path.join(VID, "packages_belt.mp4"),
   "orient": "horizontal", "line_pos": 0.60,
   "polygon": [(0.28, 0.32), (0.60, 0.32), (0.93, 0.98), (0.10, 0.98)],
   "queries": ["a cardboard box with a shipping label",
               "a package with a barcode",
               "kiện hàng trên băng chuyền"]},
  {"name": "Cà chua · phân loại", "task": "tomato",
   "path": os.path.join(VID, "tomatoes_sorting.mp4"),
   "orient": "horizontal", "line_pos": 0.72,
   "polygon": [(0.04, 0.34), (0.98, 0.30), (0.98, 0.99), (0.02, 0.99)],
   "queries": ["a ripe red tomato",
               "an unripe tomato",
               "cà chua"]},
]

# NÚM tốc độ — giảm để nhanh hơn, tăng để kỹ hơn
PROC_WIDTH     = 640    # bề rộng xử lý (nhỏ = nhanh)
MAX_FRAMES     = 16     # số frame mỗi lượt
STRIDE         = 3      # lấy mỗi N frame
MAX_NEW_TOKENS = 256    # token sinh tối đa (nhỏ = nhanh)

def _preview(v, frac=0.5):
    cap = cv2.VideoCapture(v["path"]); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(n * frac)); ok, fr = cap.read(); cap.release()
    if not ok: return None
    h, w = fr.shape[:2]
    pts = np.array([(int(x * w), int(y * h)) for x, y in v["polygon"]], np.int32)
    ov = fr.copy(); cv2.fillPoly(ov, [pts], (0, 200, 0)); fr = cv2.addWeighted(ov, 0.25, fr, 0.75, 0)
    cv2.polylines(fr, [pts], True, (0, 200, 0), 3)
    y = int(v["line_pos"] * h); cv2.line(fr, (0, y), (w, y), (0, 0, 255), 2)
    return cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, len(VIDEOS), figsize=(16, 4))
for ax, v in zip(np.atleast_1d(axes), VIDEOS):
    im = _preview(v)
    if im is not None: ax.imshow(im)
    ax.set_title(v["name"] + "\n(vùng xanh + vạch đỏ)"); ax.axis("off")
plt.tight_layout(); plt.show()
print("→ Vùng/vạch ôm đúng dòng vật chưa? Lệch thì sửa polygon/line_pos rồi chạy lại cell này.")

In [ ]:
# 🧠 Nạp LocateAnything-3B MỘT LẦN (lần đầu tải ~6GB, hơi lâu)
from run_la_conveyor import build_fast_detector
detector = build_fast_detector("nvidia/LocateAnything-3B",
                               max_new_tokens=MAX_NEW_TOKENS, iou=0.5, max_boxes=60)
print("✅ Model sẵn sàng — chạy cell tiếp theo để đếm.")

In [ ]:
# ▶️ Đếm: mỗi (video × query) → gán nhãn supervision, xuất frame + video annotate
import time, math
from run_la_conveyor import run_video

os.makedirs("out_annot", exist_ok=True)
print(f"{'video':22}{'query':42}{'qua vạch':>9}{'trong vùng':>11}{'det/fr':>8}{'giây':>7}")
print("-" * 99)
shots, t_all = [], time.time()
for v in VIDEOS:
    for q in v["queries"]:
        stem  = os.path.splitext(os.path.basename(v["path"]))[0]
        qsafe = "".join(c if c.isalnum() else "_" for c in q)[:24]
        jpg   = f"out_annot/{stem}__{qsafe}.jpg"
        mp4   = f"out_annot/{stem}__{qsafe}.mp4"
        t0 = time.time()
        r = run_video(detector, v["path"], q, orient=v["orient"], line_pos=v["line_pos"],
                      polygon=v["polygon"], proc_width=PROC_WIDTH, max_frames=MAX_FRAMES,
                      stride=STRIDE, save_annotated=jpg, save_video=mp4)
        peak = getattr(r, "zone_peak", 0)
        print(f"{v['name'][:21]:22}{q[:41]:42}{r.total_crossings:>9}{peak:>11}"
              f"{r.avg_detections:>8.1f}{time.time()-t0:>7.1f}")
        shots.append((f"{v['task']} · {q}", jpg))
print("-" * 99)
print(f"Xong {len(shots)} lượt trong {time.time()-t_all:.0f}s.  "
      f"'qua vạch'=cắt vạch · 'trong vùng'=đỉnh số vật trong polygon · 'det/fr'=box/frame (NMS).")
print("Video annotate .mp4 nằm trong ./out_annot (tải về xem chuyển động).")

# hiển thị frame annotate (nhiều box nhất) mỗi lượt
cols = 3; nrows = math.ceil(len(shots) / cols)
fig, axes = plt.subplots(nrows, cols, figsize=(16, 4 * nrows))
axf = list(np.atleast_1d(axes).flat)
for ax, (title, path) in zip(axf, shots):
    if os.path.exists(path):
        ax.imshow(cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB))
    ax.set_title(title[:50], fontsize=9); ax.axis("off")
for ax in axf[len(shots):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

### Đọc kết quả
- **`det/fr`** > 0 → LocateAnything **hiểu** query và bắt được vật (kể cả mô tả khó /
  tiếng Việt) — điểm mạnh open-vocab.
- **`qua vạch`** = số vật cắt vạch (đoạn ngắn nên số nhỏ là bình thường).
- **`trong vùng`** = đỉnh số vật ĐANG trong polygon (occupancy) — chỉ số ổn định cho
  cảnh dày/đi nhanh như cà chua.
- Muốn kỹ hơn: tăng `MAX_FRAMES`, giảm `STRIDE`. Vật/vùng lệch: sửa `polygon`/`line_pos`.
- Nhãn có dấu tiếng Việt được tự bỏ dấu khi vẽ (cv2 chỉ vẽ ASCII) — vd 'cà chua' → 'ca chua'.
- LocateAnything **chậm** (~vài giây/frame) → giữ `MAX_FRAMES` nhỏ khi thử query mới.